# TF-IDF Text Classification Practice

**Topic:** Natural Language Processing &nbsp;·&nbsp; **Difficulty:** Medium

## The data (created in the next cell)

- `train_texts` (4,000 strings), `train_labels`: synthetic product reviews, label 1 = a sentiment score of 0 or higher, so exactly neutral reviews are also labeled 1.
- `test_texts` (1,200 strings): classify these. `y_test` is used by the evaluation cell.
- Reviews mix sentiment words ("great", "awful"…), neutral filler, and negators ("not", "never") that invert the next sentiment word.

## Your task

1. Vectorize with `TfidfVectorizer` or `CountVectorizer`. Comparing `ngram_range=(1,1)` against `(1,2)` shows what negation handling is worth, but only the model behind `predictions` is graded.
2. Train a linear classifier (logistic regression is ideal).
3. Set `predictions` to your 0/1 labels for `test_texts`.

## Evaluation

Test accuracy must reach the target shown in the evaluation cell.


In [ ]:
import numpy as np

POS = ["great", "excellent", "wonderful", "fantastic", "solid", "delightful", "crisp", "reliable"]
NEG = ["awful", "terrible", "disappointing", "flimsy", "broken", "sluggish", "noisy", "useless"]
FILLER = ["the", "battery", "screen", "price", "shipping", "box", "overall", "design",
          "it", "was", "and", "feels", "seems", "quality", "product", "arrived"]
NEGATORS = ["not", "never", "hardly"]

def make(n, seed):
    r = np.random.default_rng(seed)
    texts, labels = [], []
    for _ in range(n):
        label = int(r.random() < 0.5)
        words, score = [], 0
        n_words = r.integers(8, 16)
        while len(words) < n_words:
            roll = r.random()
            if roll < 0.30:
                sentiment_pool = POS if (r.random() < (0.8 if label else 0.2)) else NEG
                word = sentiment_pool[r.integers(len(sentiment_pool))]
                if r.random() < 0.33:                       # negation flips it
                    words.append(NEGATORS[r.integers(len(NEGATORS))])
                    words.append(word)
                    score += -1 if word in POS else 1
                else:
                    words.append(word)
                    score += 1 if word in POS else -1
            else:
                words.append(FILLER[r.integers(len(FILLER))])
        texts.append(" ".join(words))
        labels.append(int(score >= 0))
    return texts, np.array(labels)

train_texts, train_labels = make(4000, 501)
test_texts, y_test = make(1200, 502)
print(f"{len(train_texts)} train reviews, e.g.: \"{train_texts[0]}\"  (label {train_labels[0]})")


In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.





In [ ]:
# =====================================================================
# EVALUATION: run this cell to compare your predictions to the truth
# =====================================================================
import numpy as np

TARGET_ACCURACY = 0.8

if "predictions" not in dir():
    raise NameError("Define `predictions` in the solution cell before running this cell.")

_pred = np.asarray(predictions).ravel()
_true = np.asarray(y_test).ravel()
if _pred.shape != _true.shape:
    raise ValueError(f"`predictions` has shape {_pred.shape}, expected {_true.shape}.")

accuracy = float((_pred == _true).mean())
print("=" * 52)
print(f"Test accuracy : {accuracy:.4f}")
print(f"Target        : >= {TARGET_ACCURACY:.2f}")
print("-" * 52)
print("Per-class recall:")
for _cls in np.unique(_true):
    _mask = _true == _cls
    print(f"  class {_cls!r:>6} : {float((_pred[_mask] == _cls).mean()):.3f}   ({int(_mask.sum())} test samples)")
print("=" * 52)
if accuracy >= TARGET_ACCURACY:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your model and try again.")


## Solution

TF-IDF with `ngram_range=(1, 2)` lets the linear model learn signed weights for bigrams like 'not great', which handles the negators. Logistic regression on top does the rest.

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

vec = TfidfVectorizer(ngram_range=(1, 2))
Xtr = vec.fit_transform(train_texts)
Xte = vec.transform(test_texts)

clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, train_labels)
predictions = clf.predict(Xte)
```
